# 01. Pedestrian network inventory

This notebook reads the 92 pedestrian street networks, one GeoPackage per
Mexican metropolitan area, and produces a summary table describing each one.

The networks were extracted from OpenStreetMap and clipped to the boundaries of
the metropolitan areas delimited by SEDATU, CONAPO and INEGI. OpenStreetMap
exports each walkable way twice, once in each direction of travel, so the first
step is to collapse those pairs into a single undirected edge set. Every figure
reported afterwards refers to that undirected network.

For each metropolitan area the notebook records the number of edges before and
after de-duplication, the total network length in kilometres, the number of
street intersections, and the connected-component structure. Components matter
here because a Mexican metropolitan area is a union of whole municipalities and
frequently contains settlements physically detached from the main urban fabric.
Any part of the network lying outside the largest component can only be reached
on foot from within that same part, which constrains the walking travel times
computed later in the pipeline.

Inputs are the 92 GeoPackages under `Networks/` and the metropolitan area
crosswalk `Inputs/metros_92.csv`, which carries each area's 2020 population and
its classification in the official national typology.

Output is `Results/01_network_metrics.parquet`, one row per metropolitan area,
together with a full run report written to `Results/logs/`.

The set of network files is summarised by a short hash of their names and sizes.
That value is printed at the start and recorded in the report, so any later
result can be traced to the exact set of input files it was computed from.

Runtime is roughly fifteen minutes. Each metropolitan area is cached as it
completes, so an interrupted run resumes where it stopped.

## 1. Setup

In [ ]:
import sys, os, glob, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'module is pointing at {mx.BASE}'

import numpy as np
import pandas as pd
import geopandas as gpd

# Identifies the exact set of network files this run reads. Leave it as None on
# a first run, then paste the printed value here to pin the inputs for every
# later run, so a changed or replaced network file stops the notebook instead of
# quietly producing different numbers.
EXPECTED_FINGERPRINT = None

FP = mx.network_fingerprint()
if EXPECTED_FINGERPRINT and FP != EXPECTED_FINGERPRINT:
    raise AssertionError(
        f'network files hash to {FP}, but this notebook is pinned to '
        f'{EXPECTED_FINGERPRINT}. The input files are not the ones these '
        'results were built from.')

print('project     ', mx.BASE)
print('networks    ', mx.DIRS['networks'])
print('fingerprint ', FP)

## 2. Match network files to metropolitan areas

In [ ]:
rep = mx.Report('01_network_metrics').start()

CACHE = mx.cache_dir('cache_01_networks')
print('cache directory:', CACHE)

metros = mx.load_metros()

files = sorted(glob.glob(os.path.join(mx.DIRS['networks'], '*_pedestrian.gpkg')))
assert len(files) == 92, f'{len(files)} pedestrian networks found, expected 92'
print(f'\n{len(files)} pedestrian networks found')

stems = pd.Series([os.path.basename(f)[:-len('_pedestrian.gpkg')] for f in files])
print('example filename stems:', stems.head(3).tolist())

In [ ]:
# Filenames and crosswalk keys are matched on a form of the name with accents,
# case and separators removed, so a difference in spelling does not split one
# metropolitan area into two. A filename that matches nothing stops the run and
# names itself, rather than dropping that metropolitan area from the results.

file_slug  = mx._slug_name(stems)
metro_slug = mx._slug_name(metros['metro_key'])

assert not metro_slug.duplicated().any(), 'metro_key values are not unique'
assert not file_slug.duplicated().any(), (
    'two network files reduce to the same name: '
    f'{file_slug[file_slug.duplicated(keep=False)].tolist()}')

missing = sorted(set(file_slug) - set(metro_slug))
if missing:
    raise AssertionError(
        f'{len(missing)} network files match no entry in the crosswalk: {missing}\n'
        f'crosswalk entries look like: {sorted(metro_slug)[:5]}')

fileframe = pd.DataFrame({'slug': file_slug.values, 'path': files})
metros = metros.assign(slug=metro_slug.values)
mx.assert_join(fileframe['slug'], metros['slug'], 'network files to metros', 1.0)

## 3. Measure each network

Geometry is read without the attribute columns, which is faster and avoids
decoding text fields that are not needed here. Coordinates are reprojected to
the national metric reference system so that lengths come out in metres and are
comparable between metropolitan areas.

In [ ]:
def metro_metrics(path):
    """Edge counts, length, intersections and component structure for one area."""
    net = mx.read_gpkg(path, columns=[])
    if len(net) == 0:
        raise RuntimeError(f'{os.path.basename(path)} returned no edges')

    src_crs = str(net.crs)
    net = net.to_crs(epsg=mx.CRS_METRIC)

    raw_n = len(net)
    ded = mx.dedupe_edges(net)
    if len(ded) == 0:
        raise RuntimeError(f'{os.path.basename(path)} lost every edge in de-duplication')

    comp = mx.component_summary(ded)
    nodes = mx.intersection_nodes(ded)

    return {
        'src_crs': src_crs,
        'edges_raw': int(raw_n),
        'edges_dedup': int(len(ded)),
        'dedup_ratio': round(len(ded) / raw_n, 4),
        'length_km': round(float(ded.geometry.length.sum() / 1000.0), 1),
        'n_intersections': int(len(nodes)),
        **comp,
    }

In [ ]:
rows = []
for i, r in enumerate(fileframe.itertuples(index=False), 1):
    cpath = os.path.join(CACHE, f'{r.slug}.parquet')
    if os.path.exists(cpath):
        rows.append(pd.read_parquet(cpath).iloc[0].to_dict())
        continue
    with mx.stage(f'[{i:>2}/92] {r.slug}'):
        rec = {'slug': r.slug, **metro_metrics(r.path)}
    pd.DataFrame([rec]).to_parquet(cpath, index=False)
    rows.append(rec)

NET = pd.DataFrame(rows).merge(
    metros[['slug', 'metro_key', 'metro_name', 'tipo', 'tier', 'conapo_pop_2020']],
    on='slug', how='left')

assert len(NET) == 92, len(NET)
assert NET['metro_name'].notna().all(), NET.loc[NET.metro_name.isna(), 'slug'].tolist()
print(f'\n{len(NET)} metropolitan areas processed')

## 4. Checks

Each check states the condition that would make it fail, so that passing means
something specific was tested rather than that nothing was examined.

In [ ]:
print('--- checks ---\n')

# Fails if any metropolitan area has an empty network after de-duplication.
empty = NET.loc[NET['edges_dedup'] == 0, 'slug'].tolist()
assert not empty, f'metropolitan areas with no edges: {empty}'
print('1. every area has edges after de-duplication   ok')

# Fails if an area was exported with a single direction per way. De-duplication
# would then remove almost nothing and that area's length would be roughly half
# the comparable figure for the others.
nodedup = NET[NET['dedup_ratio'] > 0.99]
print(f'2. areas where de-duplication removed under 1%  {len(nodedup)} of 92')
if len(nodedup):
    print(nodedup[['slug', 'edges_raw', 'edges_dedup', 'dedup_ratio']]
          .to_string(index=False))

# Fails if every area is a single connected component. Mexican metropolitan
# areas are unions of whole municipalities and routinely contain detached
# settlements, so a uniform single component indicates that disconnected parts
# were discarded when the networks were extracted.
single = int((NET['n_components'] == 1).sum())
print(f'3. areas that are a single component           {single} of 92')
assert single < 92, (
    'every area is a single connected component, so detached parts of the '
    'network appear to have been dropped during extraction')

# The source projections. Lengths are computed after reprojection, so a mix of
# zones is expected and harmless, but later work that passes a bounding box to
# these files has to convert it into the file's own projection first.
print(f'4. distinct source projections                 {NET["src_crs"].nunique()}')
print('   ', sorted(NET['src_crs'].unique()))

print(f'\n5. national network length                     {NET.length_km.sum():,.0f} km')
print(f'6. national intersections                      {NET.n_intersections.sum():,}')
print(f'7. length outside the largest component        '
      f'{NET.km_outside_largest.sum():,.0f} km '
      f'({100 * NET.km_outside_largest.sum() / NET.length_km.sum():.2f}%)')
print(f'8. areas with detached components              '
      f'{int((NET.n_components > 1).sum())} of 92')

### Areas where the detached network is substantial

Walking access is computed on the network, so a block sitting in a detached
component can only reach destinations inside that same component. Where a large
share of an area's network lies outside the largest component, unreachable
origins should be expected rather than treated as an error.

In [ ]:
frag = (NET.loc[NET['largest_share'] < 0.95,
                ['slug', 'metro_name', 'tipo', 'length_km',
                 'n_components', 'largest_share', 'km_outside_largest']]
          .sort_values('largest_share'))
print(f'{len(frag)} areas hold under 95% of their network in the largest component\n')
print(frag.to_string(index=False))
rep.add_table('areas with a substantial detached network', frag, max_rows=40)

## 5. Results

In [ ]:
cols = ['slug', 'metro_name', 'tipo', 'tier', 'conapo_pop_2020',
        'edges_raw', 'edges_dedup', 'dedup_ratio', 'length_km',
        'n_intersections', 'n_components', 'largest_share', 'km_outside_largest']

print('--- per-area network metrics, longest network first ---\n')
print(NET[cols].sort_values('length_km', ascending=False).to_string(index=False))

In [ ]:
print('\n--- by official typology (SEDATU, CONAPO and INEGI, 2020) ---\n')
by_tipo = (NET.groupby('tipo')
             .agg(areas=('slug', 'size'),
                  population=('conapo_pop_2020', 'sum'),
                  edges=('edges_dedup', 'sum'),
                  length_km=('length_km', 'sum'),
                  intersections=('n_intersections', 'sum'),
                  mean_components=('n_components', 'mean'))
             .reindex(mx.TIPO_ORDER).round(1))
print(by_tipo.to_string())

print('\n--- by population size band ---\n')
by_tier = (NET.groupby('tier')
             .agg(areas=('slug', 'size'),
                  population=('conapo_pop_2020', 'sum'),
                  length_km=('length_km', 'sum'),
                  intersections=('n_intersections', 'sum'))
             .round(1))
print(by_tier.to_string())

print('\n--- typology against size band ---\n')
print(NET.groupby(['tipo', 'tier']).size().unstack(fill_value=0).to_string())

In [ ]:
rep.add_table('per-area network metrics', NET[cols], max_rows=92)
rep.add_table('by official typology', by_tipo.reset_index(), max_rows=10)
rep.add_table('by population size band', by_tier.reset_index(), max_rows=10)
rep.add_table('network file inventory', mx.network_inventory(), max_rows=92)

## 6. Write results

In [ ]:
out = os.path.join(mx.DIRS['results'], '01_network_metrics.parquet')
NET.to_parquet(out, index=False)
print('written:', out, f'({os.path.getsize(out) / 1e6:.2f} MB)')

rep.note(f'input fingerprint {FP}; 92 metropolitan areas; '
         f'{NET.length_km.sum():,.0f} km of pedestrian network; '
         f'{NET.n_intersections.sum():,} intersections; '
         f'{100 * NET.km_outside_largest.sum() / NET.length_km.sum():.2f}% of '
         'network length outside the largest component')
rep.save()